In [0]:
import dlt
from pyspark.sql.functions import current_timestamp, col, regexp_extract, to_date, explode
from pyspark.sql.types import StructType, StructField, StringType, LongType, ArrayType, DoubleType

# ---- Explicit FRED schema ----
fred_schema = StructType([
    StructField("realtime_start", StringType(), True),
    StructField("realtime_end", StringType(), True),
    StructField("observation_start", StringType(), True),
    StructField("observation_end", StringType(), True),
    StructField("units", StringType(), True),
    StructField("output_type", LongType(), True),
    StructField("file_type", StringType(), True),
    StructField("order_by", StringType(), True),
    StructField("sort_order", StringType(), True),
    StructField("count", LongType(), True),
    StructField("offset", LongType(), True),
    StructField("limit", LongType(), True),
    StructField("observations", ArrayType(StructType([
        StructField("realtime_start", StringType(), True),
        StructField("realtime_end", StringType(), True),
        StructField("date", StringType(), True),
        StructField("value", StringType(), True),
    ])), True)
])

fred_landing_path = "abfss://lakehouse@buisdatastore.dfs.core.windows.net/bronze/fred_raw/data/"

@dlt.table(
    name="bronze_fred_raw_json",
    comment="Raw FRED API payloads as JSON (one row per file)"
)
def bronze_fred_raw_json():
    return (
        spark.readStream.format("cloudFiles")
          .option("cloudFiles.format", "json")
          .option("cloudFiles.includeExistingFiles", "true")
          .option("recursiveFileLookup", "true")
          .option("multiLine", "true")
          .schema(fred_schema)
          .load(fred_landing_path)
          .withColumn("_ingest_ts", current_timestamp())
          .withColumn("_source_file", col("_metadata.file_path"))
          .withColumn("series_id", regexp_extract(col("_metadata.file_path"), r"/data/([^/]+)/", 1))
    )

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, DoubleType
from pyspark.sql.functions import explode, col, to_date, input_file_name, current_timestamp

# ---- Google Trends paths ----
gt_landing_path = "abfss://lakehouse@buisdatastore.dfs.core.windows.net/bronze/google_trends_raw/data/"
gt_schema_path  = "abfss://lakehouse@buisdatastore.dfs.core.windows.net/bronze/_schemas/google_trends_raw/"

# ---- Explicit schema (fix for Auto Loader inference error) ----
gt_schema = StructType([
    StructField("keyword", StringType(), True),
    StructField("geo", StringType(), True),
    StructField("timeframe", StringType(), True),
    StructField("run_ts", StringType(), True),
    StructField("observations", ArrayType(StructType([
        StructField("date", StringType(), True),
        StructField("value", DoubleType(), True),
    ])), True)
])

@dlt.table(
  name="bronze_google_trends_raw",
  comment="Google Trends raw JSON exploded to one row per (keyword, date)"
)
def bronze_google_trends_raw():

    raw = (
        spark.readStream.format("cloudFiles")
          .option("cloudFiles.format", "json")
          .option("cloudFiles.schemaLocation", gt_schema_path)
          .option("cloudFiles.includeExistingFiles", "true")
          .option("recursiveFileLookup", "true")
          .schema(gt_schema)  # ✅ explicit schema
          .load(gt_landing_path)
    )

    return (
    raw.withColumn("obs", explode(col("observations")))
       .select(
           col("keyword"),
           col("geo"),
           col("timeframe"),
           to_date(col("obs.date")).alias("date"),
           col("obs.value").cast("double").alias("value"),
           col("run_ts"),
           col("_metadata.file_path").alias("source_file"),   # ✅ UC-safe
           current_timestamp().alias("ingest_ts")
       )
)

In [0]:
from pyspark.sql.functions import col, current_timestamp

landing_path = "abfss://lakehouse@buisdatastore.dfs.core.windows.net/bronze/wits_tariff_raw/data/"
schema_path  = "abfss://lakehouse@buisdatastore.dfs.core.windows.net/bronze/_schemas/wits_tariff_json_v2/"  # NEW path to force fresh schema

@dlt.table(
  name="bronze_wits_tariff_raw",
  comment="Raw WITS SDMX JSON — one row per file"
)
def bronze_wits_tariff_raw():
    return (
        spark.readStream.format("cloudFiles")
          .option("cloudFiles.format", "json")
          .option("cloudFiles.schemaLocation", schema_path)
          .option("cloudFiles.includeExistingFiles", "true")
          .option("cloudFiles.inferColumnTypes", "true")
          .option("recursiveFileLookup", "true")
          .option("multiLine", "true")
          .load(landing_path)
          .withColumn("source_file", col("_metadata.file_path"))
          .withColumn("ingest_ts", current_timestamp())
    )

In [0]:
# ============================================================
# BRONZE 4: StatCan (CSV Auto Loader)
# ============================================================
statcan_landing_path = "abfss://lakehouse@buisdatastore.dfs.core.windows.net/bronze/statcan_raw/data/"
statcan_schema_path  = "abfss://lakehouse@buisdatastore.dfs.core.windows.net/bronze/_schemas/statcan_csv/"

@dlt.table(
    name="bronze_statcan_raw",
    comment="Raw StatCan CSV data — one row per observation (all 3 tables combined). Excludes MetaData files.",
    table_properties={
        "delta.columnMapping.mode": "name",
        "delta.minReaderVersion": "2",
        "delta.minWriterVersion": "5"
    }
)
def bronze_statcan_raw():
    return (
        spark.readStream.format("cloudFiles")
          .option("cloudFiles.format", "csv")
          .option("cloudFiles.schemaLocation", statcan_schema_path)
          .option("cloudFiles.includeExistingFiles", "true")
          .option("recursiveFileLookup", "true")
          .option("header", "true")
          .option("inferSchema", "true")
          .option("cloudFiles.schemaEvolutionMode", "rescue")
          .load(statcan_landing_path)
          .where(~col("_metadata.file_path").contains("MetaData"))
          .withColumn("source_file", col("_metadata.file_path"))
          .withColumn("pid", regexp_extract(col("_metadata.file_path"), r"/data/(\d+)/", 1))
          .withColumn("ingest_ts", current_timestamp())
    )